 # Spectral Co-Adding



 Name: Isaac Anderson



 Date: 20th Nov 2025

 ### Peak Finding

 1. Use the best peak-finding tools from class to find the same 5 most prominent peaks in every channel within one data file (this may involve finding more than 5 peaks and figuring out an algorithm to find which peaks should map to which.)

##### Reading in files and necessary packages.

In [14]:
import h5py
import nbformat
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

# my utils
from utils.spectral import plot_spectra_on_histogram

# reading in channels into dataframe
filename = "./../Data/Gamma/210601_NBS295-106/20210601_152616_mass-001.hdf5"
with h5py.File(filename, "r") as hdf_file:
    channels = pd.DataFrame(
        columns=["energy"],
        index=hdf_file.keys(),
    )

    for channel_name in hdf_file:
        going_in = np.array(hdf_file[channel_name]["filt_value"])
        going_in = going_in[(0 < going_in) & (going_in < np.percentile(going_in, 97))]
        channels.loc[channel_name] = [going_in]

# setting plotly preferences
import plotly.io as pio 
pio.templates.default = "plotly_dark"

##### Histograming & columns

In [15]:
# histograming our energy
channels["counts"], channels["edges"] = zip(
    *channels["energy"].apply(np.histogram, args=(10_000,))
)
# finding the midpoints of our energy
channels["midpoints"] = [
    0.5 * (channel_edges[1:] + channel_edges[:-1])
    for channel_edges in channels["edges"]
]

##### Peak-finding with sci-py

In [16]:
peak_indices, peaks_data = map(pd.Series, zip(*channels["counts"].apply(find_peaks, prominence=4))) 
top_10_peak_locs = peaks_data.str['prominences'].apply(np.argsort).str[-8:] # "locs" is an important term here: it is the index of the indices for our argsort
channels['prominent_peak_indices'] = [
    peak_indices[channel][locs]
    for channel, locs in enumerate(top_10_peak_locs) # retrieving the original indices for each channel
]

##### Saving channels for quick use

In [17]:
# Saving popular channels for quick use
chan1 = channels.loc["chan1"]
chan45 = channels.loc["chan45"]
chan99 = channels.loc["chan99"]
display(chan1)

energy                    [1917.8376, 2934.2834, 1235.1354, 8561.532, 37...
counts                    [3, 2, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...
edges                     [0.25408345, 1.9164445, 3.5788057, 5.241167, 6...
midpoints                 [1.085264, 2.747625, 4.4099865, 6.0723476, 7.7...
prominent_peak_indices     [5063, 3837, 8392, 4202, 4221, 4304, 2890, 4473]
Name: chan1, dtype: object

##### Making sense of this graphically

In [18]:
plot_spectra_on_histogram(chan1['counts'], chan1['midpoints'], chan1['prominent_peak_indices']).show()
plot_spectra_on_histogram(chan45['counts'], chan45['midpoints'], chan45['prominent_peak_indices']).show()

2. Fit these peaks with a Gaussian on top of a linear background.


 ### Traditional Analysis

 3. Using splines with 5 peaks, co-add all the channels within one data file.

 4. Fit the most prominent peak of each individual spectrum after scaling it. Divide the Gaussian mean by the Gaussian width ($\sigma$) and histogram this quantity (which we will refer to as the signal to noise ratio or SNR).

 5. Add up all the spectra and fit the most prominent peak of the summed spectrum. Plot the SNR as a vertical dashed line on the SNR histogram from #2.

 6. Repeat steps 2 and 3 for a peak that is 2 orders of magnitude smaller (i.e. 100 times less area)

 ### DTW Analysis

 7. Use the DTW approach on all the channels within one data file to co-add them.

 8. Fit the most prominent peak of each individual spectrum after scaling it. Divide the Gaussian mean by the Gaussian width ($\sigma$) and histogram this quantity (which we will refer to as the signal to noise ratio or SNR).

 9. Add up all the spectra and fit the most prominent peak of the summed spectrum. Plot the SNR as a vertical dashed line on the SNR histogram from #2.

 10. Repeat steps 2 and 3 for a peak that is 2 orders of magnitude smaller (i.e. 100 times less area)

 ## Side Quest -- DTW Optimization



 Repeat steps 7-10 and optimize the various DTW options:

 ```

 alignment_windowed = dtw(s1, s2, keep_internals=True,

                          window_type="sakoechiba", window_args={'window_size': 2})

 ```

 ## Side Quest -- Wavelets for Drift Correction



 Inverse of noise reduction. We're keeping the noise, but removing the slow time constant terms!